In [1]:
from zk_normal import *

from __future__ import annotations

import math

In [2]:
def poseidon_hash_2(left: str, right: str) -> str:
    """
    Call the Node.js bridge to hash two child nodes using Poseidon.
    """
    cmd = f"node {FILE_PATH['commitment']} {left} {right}"
    result = subprocess.run(
        cmd, 
        shell=True, 
        capture_output=True, 
        text=True, 
        check=True
    )
    return result.stdout.strip()

In [3]:
class ZKMerkleTree:
    def __init__(self, leaves: list, depth: int):
        """
        Initialize the Merkle Tree with a fixed depth.
        Empty leaf slots will be padded with "0".
        """
        self.depth = depth
        self.max_leaves = 2 ** depth
        
        if len(leaves) > self.max_leaves:
            raise ValueError("Too many leaves for the given tree depth.")
            
        # Pad the leaves with "0" to make it a perfect binary tree
        self.leaves = leaves.copy()
        while len(self.leaves) < self.max_leaves:
            self.leaves.append("0")
            
        self.tree = [self.leaves]
        self._build_tree()

    def _build_tree(self):
        """
        Build the tree from bottom to top using Poseidon hash.
        """
        current_level = self.leaves
        for level in range(self.depth):
            next_level = []
            # Process pairs of nodes
            for i in range(0, len(current_level), 2):
                left = current_level[i]
                right = current_level[i+1]
                
                # If both are empty "0", we can optimize or just hash them
                # For strict ZKP consistency, we hash everything
                hashed_parent = poseidon_hash_2(left, right)
                next_level.append(hashed_parent)
                
            self.tree.append(next_level)
            current_level = next_level

    def get_root(self) -> str:
        """
        Return the global root of the Merkle Tree.
        """
        return self.tree[-1][0]

    def get_path(self, index: int) -> dict:
        """
        Generate the Merkle path for a specific leaf index.
        Returns the exact format required by Circom inputs.
        """
        path_elements = []
        path_indices = []
        
        current_index = index
        for level in range(self.depth):
            is_right_node = current_index % 2 != 0
            sibling_index = current_index - 1 if is_right_node else current_index + 1
            
            # Add the sibling's value to the path elements
            path_elements.append(self.tree[level][sibling_index])
            
            # Circom needs 1 if the node is on the right, 0 if on the left
            path_indices.append(1 if is_right_node else 0)
            
            # Move to the parent node's index for the next level
            current_index //= 2
            
        return {
            "path_elements": path_elements,
            "path_indices": path_indices
        }
    # Generate 31 bytes random secret for a voter
    def generate_voter_secret() -> str:
        return os.urandom(31).hex()

    # Call Node.js to compute Poseidon Hash
    def compute_commitment(voter_id_hash: str, secret_hex: str) -> str:
        voter_id_int = str(int(voter_id_hash, 16))
        secret_int = str(int(secret_hex, 16))
        
        cmd = f"node commitment.js {voter_id_int} {secret_int}"
        result = subprocess.run(
            cmd, 
            shell=True, 
            capture_output=True, 
            text=True, 
            check=True
        )
        return result.stdout.strip()

In [4]:
def calculate_optimal_depth(num_voters: int) -> int:
    if num_voters <= 1:
        return 1
    return math.ceil(math.log2(num_voters))



In [5]:
if __name__ == "__main__":
    # 1. Read voters from CSV
    voters = readCsvData(FILE_PATH['csv'])
    num_voters = len(voters)
    
    # 2. Auto-calculate tree depth
    optimal_depth = calculate_optimal_depth(num_voters)
    max_capacity = 2 ** optimal_depth
    
    print(f"Total voters: {num_voters}")
    print(f"Optimal tree depth: {optimal_depth} (Capacity: {max_capacity})")
    print("-" * 30)
    
    registered_commitments = []
    voter_secrets = {}
    
    # 3. Generate secrets and commitments from scratch
    for voter in voters:
        secret = ZKMerkleTree.generate_voter_secret()
        voter_secrets[voter.id] = secret  # Store securely for later proving phase
        
        commitment = ZKMerkleTree.compute_commitment(voter.hashId, secret)
        registered_commitments.append(commitment)
        
        print(f"Registered User {voter.id} -> Commitment: {commitment[:10]}...")
    
    # 4. Build the Merkle Tree
    print("-" * 30)
    print("Building Merkle Tree...")
    merkle_tree = ZKMerkleTree(registered_commitments, optimal_depth)
    
    global_root = merkle_tree.get_root()
    print(f"Global Root established: {global_root}")

Total voters: 84
Optimal tree depth: 7 (Capacity: 128)
------------------------------
Registered User S1242121 -> Commitment: 8174821392...
Registered User S1331116 -> Commitment: 1368121727...
Registered User M1452004 -> Commitment: 1511165456...
Registered User M1452019 -> Commitment: 9672394366...
Registered User M1454007 -> Commitment: 1924024307...
Registered User S1078013 -> Commitment: 1365897447...
Registered User S1161130 -> Commitment: 3725422141...
Registered User S1253041 -> Commitment: 1096735098...
Registered User S1278104 -> Commitment: 2081433809...
Registered User S1178010 -> Commitment: 8298375743...
Registered User S1178113 -> Commitment: 1454420347...
Registered User S1331001 -> Commitment: 8436473790...
Registered User S1111025 -> Commitment: 1203083266...
Registered User S1125015 -> Commitment: 6590631124...
Registered User S1267021 -> Commitment: 5915709991...
Registered User S1161010 -> Commitment: 2239362160...
Registered User S1263061 -> Commitment: 4721254577